### TOOLS

Models can request to call tools that perform tasks such as fetching data from databases, searching the web, or running code. Tools are pairings of:
    1.A schema including the name of the tool, a description, and/or argument definitions(often a JSON schema)
    2.A function or coroutine to execute 

In [5]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3.6-27b",
                        max_tokens=400)
response = model.invoke("WHy do parrots talk?")
response


AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Question**: The user asks "WHy do parrots talk?" (Note: typo "WHy" -> "Why"). The core question is about the biological, evolutionary, and behavioral reasons behind parrots\' ability to mimic human speech.\n\n2.  **Identify Key Concepts**:\n   - Parrot vocal learning/mimicry\n   - Evolutionary purpose (social bonding, communication within flocks)\n   - Brain structure (specialized vocal learning centers)\n   - Human interaction context (why they learn human speech specifically)\n   - Not "talking" in the human sense (lack of semantic understanding)\n\n3.  **Structure the Response**:\n   - Clarify the misconception (they mimic, don\'t "talk" with human-level comprehension)\n   - Explain the biological/evolutionary reasons\n   - Describe the neural mechanisms\n   - Explain why they learn human speech specifically\n   - Keep it clear, scientific but accessible\n   - Address the typo implicitly by answering 

In [6]:
from langchain.tools import tool

@tool
def get_weather(location:str) -> str:
    """Get weather at a location"""
    return f"Its sunny in {location}"

model_with_tools = model.bind_tools([get_weather])


In [7]:
response = model_with_tools.invoke("whats the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s request: "whats the weather like in Boston?"\n2.  Identify the required tool: `get_weather`\n3.  Extract parameters: `location` = "Boston"\n4.  Call the tool.\n5.  Formulate response based on tool output. (Wait for tool output first)\nLet\'s call the tool. \n`get_weather(location="Boston")`✅\n', 'tool_calls': [{'id': 'qpczw9fct', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 122, 'prompt_tokens': 275, 'total_tokens': 397, 'completion_time': 0.233098322, 'completion_tokens_details': {'reasoning_tokens': 94}, 'prompt_time': 0.018690053, 'prompt_tokens_details': None, 'queue_time': 0.048171732, 'total_time': 0.251788375}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_49d6b1859d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': '

### Tool Execution Loops

In [8]:
#step 1 : Model genration tool calls
messages = [{"role":"user","content":"what's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#step 2: Execute the tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#step 3: Pass results back to model for final responses
final_response = model_with_tools.invoke(messages)
print(final_response.text)
#"The current weather in Boston is 72F and sunny."


It's sunny in Boston.


In [9]:
messages

[{'role': 'user', 'content': "what's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s intent: The user wants to know the weather in Boston.\n2.  Identify the required tool: `get_weather`\n3.  Extract parameters: location = "Boston"\n4.  Call the tool: `get_weather(location="Boston")`\n5.  Format the response based on the tool\'s output. (I will simulate the tool call and then provide the response based on typical weather data, but since I\'m an AI, I\'ll just make the tool call as instructed). Wait, I need to actually output the tool call.\n\nLet\'s construct the tool call.\nFunction: `get_weather`\nParameter: `location`: "Boston"\nRequired parameters are met.\nProceed. \nOutput matches the expected format.✅\n', 'tool_calls': [{'id': '00wep0b6j', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 194,